# Phase 3a: LLM explanations and stance labels (run once)

**Settings:** Accelerator **GPU T4 x2** (this notebook uses **both** GPUs) and **Internet on**.

Qwen2.5-7B-Instruct reads every gold parent→reply pair (4,751 in total) and writes
`/kaggle/working/llm/llm_outputs.jsonl` with the reply's stance toward the parent and a short explanation.
The file is saved after every batch. If the session stops, open the notebook again, attach **this notebook's last version**
as an input, and run it again. It continues where it stopped.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
# Smoke test (~5-10 min, mostly the 15 GB model download): 5 pairs per split. Check that parse_ok is true for most lines.
!python generate_llm.py --limit 5 --batch_size 8 --out /kaggle/working/llm_smoke/llm_outputs.jsonl
!head -n 3 /kaggle/working/llm_smoke/llm_outputs.jsonl

In [ ]:
# Full run. Earlier partial outputs attached as inputs are continued, not regenerated.
prev = sorted(glob.glob("/kaggle/input/*/llm/llm_outputs.jsonl") + glob.glob("/kaggle/input/*/*/llm/llm_outputs.jsonl"))
print("Continuing from:", prev or "nothing")
!python generate_llm.py --out /kaggle/working/llm/llm_outputs.jsonl --resume_from {" ".join(prev)}

In [ ]:
# Sanity check: parse rate, and stance distribution per gold label (counter replies should lean to "oppose").
import json, collections
llm = [json.loads(l) for l in open("/kaggle/working/llm/llm_outputs.jsonl")]
gold = {(s, json.loads(l)["idx"]): int(json.loads(l)["label"]) for s in ("train", "val", "test")
        for l in open(f"data/counter_context/data/gold/{s}.jsonl")}
names = {0: "hate", 1: "neutral", 2: "counter"}
print(f"{len(llm)} outputs, parsed cleanly: {sum(o['parse_ok'] for o in llm) / len(llm):.1%}")
table = collections.Counter((names[gold[(o['split'], o['idx'])]], o["stance"]) for o in llm)
for lab in names.values():
    row = {s: table[(lab, s)] for s in ("support", "oppose", "neutral")}
    print(f"{lab:8s}", row)